<a href="https://colab.research.google.com/github/houstongolden/bigbounce/blob/main/research/notebooks/bigbounce_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BigBounce GPU Research Notebook

**Purpose:** Run GPU-intensive models (Polymathic AI Walrus/AION) and large-scale astronomical ML on Google Colab Pro.

**Setup:** Runtime → Change runtime type → GPU (A100 recommended)

**Secrets:** Add these in the Colab sidebar (key icon):
- `HUGGINGFACE_TOKEN` — for gated model access
- `ANTHROPIC_API_KEY` — for Claude reasoning calls
- `GOOGLE_AI_API_KEY` — for Gemini multimodal
- `DEEPSEEK_API_KEY` — for DeepSeek R1 verification

**Repo:** [Hubify-Projects/bigbounce](https://github.com/Hubify-Projects/bigbounce)

In [ ]:
# ── 0. Install dependencies ─────────────────────────────────────────
!pip install -q transformers torch datasets astroML astroquery \
    anthropic openai google-generativeai requests python-dotenv pandas numpy scipy matplotlib

In [ ]:
# ── 1. Load API keys from Colab Secrets ─────────────────────────────
import os
try:
    from google.colab import userdata
    os.environ['HUGGINGFACE_TOKEN'] = userdata.get('HUGGINGFACE_TOKEN')
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['GOOGLE_AI_API_KEY'] = userdata.get('GOOGLE_AI_API_KEY')
    os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
    print('Loaded keys from Colab Secrets')
except Exception:
    print('Not in Colab or secrets not set — using environment variables')

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({mem:.1f} GB)')
else:
    print('WARNING: No GPU detected. Change runtime to GPU.')

In [ ]:
# ── 2. Clone BigBounce repo (for research agents) ───────────────────
!git clone https://github.com/Hubify-Projects/bigbounce.git /content/bigbounce 2>/dev/null || echo 'Already cloned'
import sys
sys.path.insert(0, '/content/bigbounce')

# Import research agents
from research.agents.dataset_loaders import (
    load_mmu, load_astroml, load_polymathic,
    list_mmu_datasets, list_astroml_datasets, list_polymathic_models
)
print('Research agents loaded')
print(f'MMU datasets: {list(list_mmu_datasets().keys())}')
print(f'Polymathic models: {list_polymathic_models()}')

## Polymathic AI Models

Physics foundation models trained on simulation data. Useful for:
- **Walrus (1.3B):** Simulation-based inference on physical systems
- **AION-base (0.3B):** Astronomical omnimodal network — fast inference
- **AION-large (3.1B):** Full astronomical omnimodal network — highest quality

GPU requirements: Walrus needs ~4GB VRAM, AION-large needs ~12GB.

In [ ]:
# ── 3. Load Polymathic AI — Walrus (1.3B) ───────────────────────────
# Requires GPU with 4GB+ VRAM
from transformers import AutoModel

hf_token = os.environ.get('HUGGINGFACE_TOKEN', '')

print('Loading Walrus (1.3B params)...')
walrus = AutoModel.from_pretrained(
    'polymathic-ai/walrus',
    token=hf_token if hf_token else None,
    device_map='auto',
    trust_remote_code=True,
)
print(f'Walrus loaded on {next(walrus.parameters()).device}')
print(f'Parameters: {sum(p.numel() for p in walrus.parameters()):,}')

## Multimodal Universe Datasets

100TB of astronomical data on HuggingFace. Stream without downloading.

In [ ]:
# ── 4. Stream PLAsTiCC light curves ─────────────────────────────────
plasticc = load_mmu('plasticc', streaming=True, max_samples=100)

# Peek at first sample
sample = next(iter(plasticc))
print(f'PLAsTiCC sample keys: {list(sample.keys())}')
print(f'Target: {sample.get("target", "N/A")}')

In [ ]:
# ── 5. Stream JWST CEERS galaxy images ──────────────────────────────
ceers = load_mmu('jwst_ceers', streaming=True, max_samples=10)

sample = next(iter(ceers))
print(f'JWST CEERS sample keys: {list(sample.keys())}')

In [ ]:
# ── 6. Load SDSS spectra via AstroML ────────────────────────────────
spectra = load_astroml('sdss_spectra')
print(f'SDSS spectra shape: {spectra.shape}')
print(f'Columns: {spectra.dtype.names[:10]}...')

## MAST / JWST Data Access

Query JWST observations directly from Colab. Free, no auth needed.

In [ ]:
# ── 7. Query JWST observations ──────────────────────────────────────
from research.agents.data_access import search_jwst, search_gaia

# Search for NGC 1365 JWST observations
obs = search_jwst('NGC 1365', radius_arcmin=5)
print(f'Found {len(obs)} JWST observations of NGC 1365')
for o in obs[:5]:
    print(f"  {o['obs_id']}: {o['instrument']} {o['filters']} ({o['exposure_time']:.0f}s)")

In [ ]:
# ── 8. Query Gaia DR3 ───────────────────────────────────────────────
# Stars near Galactic Center
stars = search_gaia(ra=266.4, dec=-29.0, radius_deg=0.1, max_results=20)
print(f'Found {len(stars)} Gaia DR3 sources near Galactic Center')
for s in stars[:5]:
    print(f"  Gaia {s['source_id']}: G={s['phot_g_mean_mag']:.2f}, parallax={s['parallax']:.3f} mas")

## BigBounce-Specific Analysis

Templates for research tasks directly relevant to the paper.

In [ ]:
# ── 9. Template: CMB parity analysis data retrieval ──────────────────
# Fetch Planck CMB data products from MAST/ESA
# This is a starting point — adapt for specific analysis needs

from research.agents.data_access import search_mast

# Search Planck observations
planck = search_mast(target='Planck', collection='Planck')
print(f'Planck products in MAST: {len(planck)}')

In [ ]:
# ── 10. Template: Cross-check equations with DeepSeek R1 ─────────────
from research.agents.computation import deepseek_verify

result = deepseek_verify(
    claim="The inflationary suppression factor D_inf = e^{-3N}(T_reh/M_GUT)^{3/2} "
          "gives D_inf ~ 10^{-121} for N=92, T_reh=10^{15} GeV, M_GUT=10^{16} GeV",
    context="Einstein-Cartan cosmology with Holst term, evaluating parity-odd operator "
            "on-shell during inflation"
)
print(f"Verdict: {result['verdict']}")
print(f"\n{result['content'][:1000]}")